# Siren Classifier FPGA Inference — PYNQ-Z2

**Board:** PYNQ-Z2 (Zynq XC7Z020)  
**Model:** 2-layer MLP  `41 → 4 → 2`  
**Task:** Audio siren classification — ambulance vs firetruck  
**Classes:** ambulance (0), firetruck (1)

---

### Files required in this folder

| File | Source |
|------|--------|
| `mlp_accelerator.bit` | Vivado output → `fpga/vivado/create_project.tcl` |
| `mlp_accelerator.hwh` | Extracted from `mlp_accelerator.xsa` via `fpga/pynq/extract_hwh.py` |
| `feature_stats.npz` | Generated by `fpga/hls/generate_weights.py` |

### Stream protocol (ap_ctrl_none — free-running)
- **Send:** 41 uint32 words — one Q4.12 int16 feature in bits[15:0] per word (164 bytes total)
- **Recv:** 1 uint32 word — argmax class index 0 (ambulance) or 1 (firetruck)
- **Usage:** `dma.sendchannel.transfer(in_buf)` → `dma.recvchannel.transfer(out_buf)` → `wait()` × 2

No `hls_ip.write(0x00, 0x01)` required — the kernel runs as a free-running streaming pipeline.

In [ ]:

# ============================================================
# CELL 1: Imports & Constants
# ============================================================
import numpy as np
import time
import io
import os
from pynq import Overlay, allocate
from scipy.io import wavfile
from scipy.signal import resample, spectrogram, hann as scipy_hann
import ipywidgets as widgets
from IPython.display import display

# -- Resolve asset directory from INTEGRATION_BASE_DIR env var --
_base    = os.environ.get("INTEGRATION_BASE_DIR", "").strip()
if not _base:
    _base = os.path.abspath(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))
_mlp_dir = os.path.join(_base, "models", "mlp")

# -- Model-specific constants (must match mlp_top.h and train_radioml.py) --
N_IN   = 41         # input features (spectral + waveform)
N_OUT  = 2          # output classes
SCALE  = 4096.0     # Q4.12: 1 << 12

CLASSES   = ['ambulance', 'firetruck']
BIT_FILE  = os.path.join(_mlp_dir, 'mlp_accelerator.bit')
STATS_FILE = os.path.join(_mlp_dir, 'feature_stats.npz')

# -- Feature extraction constants (must match training) --
TARGET_SR    = 16000
NPERSEG      = 512
NOVERLAP     = 256
SPEC_BANDS   = [(0,500),(500,1000),(1000,2000),(2000,4000),
                (4000,6000),(6000,8000),(8000,10000),(10000,16000)]
ENERGY_BANDS = [(0,300),(300,1000),(1000,3000),(3000,8000)]

print("Constants defined.")
print(f"  Asset dir      : {_mlp_dir}")
print(f"  Input size     : {N_IN}")
print(f"  Output classes : {N_OUT}  -> {CLASSES}")
print(f"  Fixed-point    : Q4.12  (SCALE = {int(SCALE)})")
print(f"  Bitstream      : {BIT_FILE}")


In [ ]:

# ============================================================
# CELL 2: Load Feature Stats + Bitstream onto FPGA
# ============================================================
# -- Feature normalisation stats --
if os.path.exists(STATS_FILE):
    s = np.load(STATS_FILE)
    feat_mean = s['mean'].astype(np.float32)
    feat_std  = s['std'].astype(np.float32)
    print(f"Loaded feature_stats.npz  (mean[:4] = {feat_mean[:4]})")
else:
    feat_mean = np.zeros(N_IN, dtype=np.float32)
    feat_std  = np.ones(N_IN,  dtype=np.float32)
    print(f"WARNING: feature_stats.npz not found at {STATS_FILE} -- using zero-mean/unit-std")

# -- Bitstream --
if not os.path.exists(BIT_FILE):
    raise FileNotFoundError(
        f"Required file not found: {BIT_FILE}\n"
        f"  Expected at: {_mlp_dir}\n"
        "  Ensure deploy_bundle/models/mlp/ contains mlp_accelerator.bit"
    )

print(f"Loading bitstream: {BIT_FILE} ...")
overlay = Overlay(BIT_FILE)
print("Bitstream loaded successfully!")
print("\nAvailable IP blocks:", list(overlay.ip_dict.keys()))


In [ ]:
# ============================================================
# CELL 3: Acquire DMA Handle
# ============================================================
dma = overlay.axi_dma_0

print('DMA handle:', dma)
print('Kernel is free-running -- no ap_start write needed.')

In [ ]:
# ============================================================
# CELL 4: Preprocessing & Inference Function
# ============================================================

def extract_features(wav_path):
    """Extract 41-dim feature vector from a WAV file."""
    sr, data = wavfile.read(wav_path)
    if data.dtype == np.int16:
        data = data.astype(np.float32) / 32768.0
    elif data.dtype == np.int32:
        data = data.astype(np.float32) / 2147483648.0
    else:
        data = data.astype(np.float32)
    if data.ndim == 2:
        data = data.mean(axis=1)
    peak = np.abs(data).max()
    if peak > 1e-8:
        data /= peak
    if sr != TARGET_SR:
        data = resample(data, int(len(data) * TARGET_SR / sr))
    freqs, _, Sxx = spectrogram(
        data, fs=TARGET_SR, window=scipy_hann(NPERSEG),
        nperseg=NPERSEG, noverlap=NOVERLAP, scaling='spectrum', mode='magnitude')
    Sxx = np.log1p(Sxx)
    feats = []
    for flo, fhi in SPEC_BANDS:
        m = (freqs >= flo) & (freqs < fhi)
        b = Sxx[m, :].flatten() if m.any() else np.zeros(1)
        feats += [np.mean(b), np.std(b), np.percentile(b, 25), np.percentile(b, 75)]
    abs_d = np.abs(data)
    feats += [np.mean(data), np.std(data), np.mean(abs_d), np.max(abs_d), np.percentile(abs_d, 90)]
    for flo, fhi in ENERGY_BANDS:
        m = (freqs >= flo) & (freqs < fhi)
        bm = Sxx[m, :].mean(axis=0) if m.any() else np.zeros(1)
        feats.append(float(np.mean(bm)))
    return np.array(feats, dtype=np.float32)


def preprocess(feats):
    """Normalise features and quantise to Q4.12 uint32 words."""
    x = (np.asarray(feats, dtype=np.float32) - feat_mean) / (feat_std + 1e-8)
    q = np.clip(np.round(x * SCALE), -32768, 32767).astype(np.int16)
    return q.astype(np.uint16).astype(np.uint32)


def run_inference(feats):
    """
    Run one forward pass through the FPGA MLP accelerator.

    Parameters
    ----------
    feats : array-like, shape (41,) -- normalised float32 features

    Returns
    -------
    pred : int   -- predicted class index (0=ambulance, 1=firetruck)
    ms   : float -- DMA round-trip latency in ms
    """
    in_buf  = allocate(shape=(N_IN,), dtype=np.uint32)
    out_buf = allocate(shape=(1,),    dtype=np.uint32)
    np.copyto(in_buf, preprocess(feats))
    t0 = time.perf_counter()
    dma.sendchannel.transfer(in_buf)
    dma.recvchannel.transfer(out_buf)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    t1 = time.perf_counter()
    pred = int(out_buf[0]) & 0x1
    in_buf.freebuffer()
    out_buf.freebuffer()
    return pred, (t1 - t0) * 1000.0


print("Preprocessing and inference functions ready.")

In [ ]:
# ============================================================
# CELL 5: Single-Sample Inference Test (Random Input)
# ============================================================
print("=" * 55)
print("  Single Inference Test -- Random Feature Vector")
print("=" * 55)

x_test = np.random.uniform(-2.0, 2.0, N_IN).astype(np.float32)

pred, ms = run_inference(x_test)

class_name = CLASSES[pred] if 0 <= pred < len(CLASSES) else f"Unknown({pred})"

print(f"Prediction  : Class {pred}  ({class_name})")
print(f"Latency     : {ms:.3f} ms  (end-to-end incl. DMA)")
print()
print("(Random input -- prediction has no ground-truth meaning here.)")

In [ ]:

# ============================================================
# CELL 6: Dispatch from ROUTED_INPUT_PATH (or interactive upload)
# ============================================================
_routed = os.environ.get("ROUTED_INPUT_PATH", "").strip()

if _routed:
    # ── Dispatched mode: file path injected by the integration dispatcher ──
    print("=" * 60)
    print("  Siren MLP FPGA Classifier -- Dispatched Inference")
    print("=" * 60)
    print(f"Input file : {_routed}")
    try:
        feats = extract_features(_routed)
        pred, ms = run_inference(feats)
        class_name = CLASSES[pred] if 0 <= pred < len(CLASSES) else f"Unknown({pred})"
        print(f"\nPrediction : {class_name.upper()}  (class {pred})")
        print(f"Latency    : {ms:.3f} ms")
    except Exception as _e:
        import traceback
        print(f"Error during inference: {_e}")
        traceback.print_exc()

else:
    # ── Interactive mode: FileUpload widget ────────────────────────────────
    print("=" * 60)
    print("  Siren MLP FPGA Classifier -- Upload WAV File")
    print("=" * 60)

    upload_btn  = widgets.FileUpload(accept='.wav', multiple=False,
                                     description='Upload WAV',
                                     layout=widgets.Layout(width='180px'))
    out_widget  = widgets.Output()

    def on_upload_change(change):
        with out_widget:
            out_widget.clear_output()
            val = upload_btn.value
            if isinstance(val, dict):
                fname   = list(val.keys())[0]
                content = val[fname]['content']
            else:
                item    = list(val)[0]
                fname   = item.get('name', 'audio.wav') if isinstance(item, dict) else getattr(item, 'name', 'audio.wav')
                content = item.get('content', None)     if isinstance(item, dict) else getattr(item, 'content', None)
                if content is None:
                    content = bytes(item)
            tmp = '/tmp/siren_upload.wav'
            with open(tmp, 'wb') as f:
                f.write(bytes(content) if not isinstance(content, bytes) else content)
            try:
                feats = extract_features(tmp)
                pred, ms = run_inference(feats)
                class_name = CLASSES[pred]
                print(f"Prediction : {class_name.upper()}  (class {pred})")
                print(f"File       : {fname}")
                print(f"Latency    : {ms:.3f} ms")
            except Exception as e:
                import traceback
                print(f"Error: {e}")
                traceback.print_exc()

    upload_btn.observe(on_upload_change, names='value')
    display(
        widgets.HTML("<h3>Siren MLP Hardware Classifier</h3>"
                     "<p>Upload a <b>.wav</b> file of a siren recording.<br>"
                     "The FPGA MLP classifies it as <b>ambulance</b> or <b>firetruck</b>.</p>"),
        upload_btn, out_widget
    )


In [ ]:
# ============================================================
# CELL 7: Batch Throughput Benchmark -- 1000 Inferences
# ============================================================
print("=" * 60)
print("  Batch Benchmark -- 1000 Random Feature Vectors")
print("=" * 60)

N_BENCH = 1000
xs      = np.random.uniform(-2.0, 2.0, (N_BENCH, N_IN)).astype(np.float32)

latencies   = []
predictions = []

t_wall_start = time.perf_counter()
for i in range(N_BENCH):
    pred, ms = run_inference(xs[i])
    latencies.append(ms)
    predictions.append(pred)
t_wall_end = time.perf_counter()

latencies = np.array(latencies)
total_ms  = (t_wall_end - t_wall_start) * 1000.0

print(f"Samples Processed  : {N_BENCH}")
print(f"Avg System Latency : {latencies.mean():.3f} ms  (incl. DMA overhead)")
print(f"Std Dev            : {latencies.std():.3f} ms")
print(f"Min / Max          : {latencies.min():.3f} ms / {latencies.max():.3f} ms")
print(f"P5  / P95          : {np.percentile(latencies, 5):.3f} ms / {np.percentile(latencies, 95):.3f} ms")
print(f"Throughput         : {1000.0 * N_BENCH / total_ms:.0f} inferences/sec")
print()

from collections import Counter
counts = Counter(CLASSES[p] if 0 <= p < len(CLASSES) else f"Unknown({p})" for p in predictions)
print("Prediction distribution (random input -- expect roughly 50/50):")
for cls in CLASSES:
    cnt = counts.get(cls, 0)
    pct = 100.0 * cnt / N_BENCH
    bar = '#' * int(pct / 2)
    print(f"  {cls:<12} : {cnt:4d} ({pct:5.1f}%)  {bar}")

In [ ]:
# ============================================================
# CELL 8: Debug -- DMA & Overlay Inspection
# ============================================================
print("Overlay IP dict:")
for name, info in overlay.ip_dict.items():
    print(f"  {name:<30} : {info.get('type', '?')}")

print()
print("DMA object:", dma)
print()
print("DMA channel status:")
try:
    print(f"  sendchannel  : running={dma.sendchannel.running}")
    print(f"  recvchannel  : running={dma.recvchannel.running}")
except Exception as e:
    print(f"  (could not read channel status: {e})")

print()
print("If inference hangs: reload the overlay with Overlay(BIT_FILE)")
print("  overlay = Overlay('mlp_accelerator.bit')")
print("  dma = overlay.axi_dma_0")